In [1]:
from google.cloud import storage
import pandas as pd
from io import BytesIO

# Initialize Google Cloud Storage client
storage_client = storage.Client()

# Read first Parquet file
bucket_name = "unt_capstone_project_2025"
file_path1 = "business_reviews_merged.parquet"
file_path2 = "user_subset.parquet"

bucket = storage_client.bucket(bucket_name)

# Load first file
blob1 = bucket.blob(file_path1)
data1 = blob1.download_as_bytes()
business_reviews_merged_df = pd.read_parquet(BytesIO(data1))

# Load second file
blob2 = bucket.blob(file_path2)
data2 = blob2.download_as_bytes()
user_subset_df = pd.read_parquet(BytesIO(data2))

In [2]:
user_subset_df['elite'] = user_subset_df['elite'].astype(str)

In [3]:
# Apply a row-wise operation to find the max year for each row
user_subset_df['yelping_end_year'] = (
    user_subset_df['elite']
    .apply(lambda x: max(map(int, x.split(','))) if x not in ['', 'nan'] else None)
)

In [4]:
# Convert the 'elite' column into a list by splitting the comma-separated values
user_subset_df['elite'] = user_subset_df['elite'].astype(str).apply(lambda x: x.split(',') if x != 'nan' else [])

In [5]:
# Use explode() to expand the values into multiple rows
user_subset_df = user_subset_df.explode('elite')

In [6]:
user_subset_df = user_subset_df.replace({'elite': {'20': '2020'}})

In [7]:
# Ensure 'yelping_since' is in datetime format
user_subset_df['yelping_since'] = pd.to_datetime(user_subset_df['yelping_since'])

In [8]:
# Extract the year
user_subset_df['yelping_start_year'] = user_subset_df['yelping_since'].dt.year

In [9]:
user_subset_df.rename(columns={'useful': 'user_sent_useful', 'funny': 'user_sent_funny', 'cool': 'user_sent_cool'}, inplace=True)

In [10]:
business_reviews_merged_df['date'] = pd.to_datetime(business_reviews_merged_df['date'])

In [11]:
# Extract the year
business_reviews_merged_df['review_year'] = business_reviews_merged_df['date'].dt.year

In [12]:
business_reviews_merged_df['review_year'] = business_reviews_merged_df['review_year'].astype(str)

In [13]:
business_reviews_merged_df = business_reviews_merged_df.replace({'review_year': {'2018.0': '2018','2019.0': '2019', '2017.0': '2017','2021.0': '2021', '2020.0': '2020','2022.0': '2022', 'nan': '2016', '2016.0':'2016'}})

In [14]:
def merge_dataframes(left_df, right_df, left_ids, right_ids, merge_type='left'):
    """
    Merges two DataFrames based on specified ID columns.

    Args:
        left_df (pd.DataFrame): The left DataFrame.
        right_df (pd.DataFrame): The right DataFrame.
        left_ids (list): List of column names to use as IDs from the left DataFrame.
        right_ids (list): List of column names to use as IDs from the right DataFrame.
        merge_type (str, optional): Type of merge to perform ('inner', 'left', 'right', 'outer'). Defaults to 'inner'.

    Returns:
        pd.DataFrame: The merged DataFrame.
    """
    merged_df = pd.merge(left_df, right_df, left_on=left_ids, right_on=right_ids, how=merge_type)
    return merged_df

In [15]:
left_id_cols = ['user_id', 'review_year']
right_id_cols = ['user_id', 'elite']

In [16]:
business_reviews_users_merged_df = merge_dataframes(business_reviews_merged_df, user_subset_df, left_id_cols, right_id_cols, merge_type='left')

In [17]:
business_reviews_users_merged_df.to_parquet('business_reviews_users_merged.parquet')